# Slug Clustering by Country Coverage

Groups slugs by their country coverage patterns using Jaccard similarity on binary presence matrices.
Missingness is signal — the absence of an indicator for a country is valuable information.

**Phase 1: Model Definition**

In [1]:
using Revise
using InteractiveUtils

includet("phase1/functions/load_phase1.jl")

╔══════════════════════════════════════════════════════════════════════════╗
║ QoG METADATA JOINING - PHASE 0 LOADED                                ║
╚══════════════════════════════════════════════════════════════════════════╝

Quick Start:
    metadata = join_metadata()              # Run full pipeline (single isomorphism check)
    metadata = join_metadata_with_cascade()  # Run cascade (strictest → loosest), then union on slug
    quick_check()                             # Diagnostic check
    inspect_exceptions()                      # Review configuration
    show_usage()                              # Detailed documentation

Pipeline Steps:
    1. ingest_and_normalize()            # Load & normalize sources (PDF = qog_slugs_temporal.csv; min_year/max_year ingested)
    2. align_id_variables!(...)          # Harmonize ID vars
    3. run_isomorphism_cascade(...)      # Strictest → loosest until success; returns (stata_df, pdf_df, arrow_df) for union on slug
    4. unify_and_join(..

In [2]:
using CSV, DataFrames

df = load_augmented_or_build()
meta_df = CSV.read("data/qog_metadata_plus2.csv", DataFrame)
geo_df = CSV.read("data/ggis_geographic_lookup.csv", DataFrame)
pool = CSV.read("data/clustering_pool.csv", DataFrame).slug

println("Loaded: $(nrow(df)) rows, $(length(pool)) slugs in clustering pool")

✓ Checksum verified: data/qog_std_ts_jan25_aug.arrow
✓ Loaded: 12391 rows × 2014 cols from data/qog_std_ts_jan25_aug.arrow
  ggis_rowid unique: ✓ | Required columns: ✓ | Missing regions: 0 ✓
Loaded: 12391 rows, 670 slugs in clustering pool


## Run Clustering Pipeline

Default thresholds — adjust as needed.

In [3]:

# Adjust thresholds:
#     result = run_slug_clustering(df, meta_df, geo_df;
#         presence_min_pct=0.15,    # stricter presence
#         min_sim=0.20,             # tighter clusters
#         k=10                      # fewer edges
#     )

# Build matrix from filtered pool (no partition_slugs needed)
matrix_result = build_slug_country_matrix(df, pool)
edges = jaccard_topk_edges(matrix_result.X, matrix_result.slug_index)
edges_und = symmetrize_edges(edges)
graph_result = build_slug_graph(edges_und)
slug_cluster_df = cluster_slugs(graph_result)

# Interpret
profiles = build_cluster_profiles(slug_cluster_df, meta_df,
    matrix_result.X, matrix_result.slug_index, matrix_result.country_index, geo_df)

ht_validation = validate_ht_region_alignment(slug_cluster_df,
    matrix_result.X, matrix_result.slug_index, matrix_result.country_index, df)


LoadError: MethodError: no method matching build_slug_country_matrix(::DataFrame, ::Vector{String31})
The function `build_slug_country_matrix` exists, but no method is defined for this combination of argument types.

[0mClosest candidates are:
[0m  build_slug_country_matrix(::DataFrame, [91m::Vector{String}[39m; presence_min_pct, presence_max_pct, min_country_coverage, verbose)
[0m[90m   @[39m [35mMain[39m [90m~/work/phase1/functions/[39m[90m[4mslug_clustering.jl:107[24m[39m


## Cluster Profiles

In [9]:
result.profiles

Row,cluster_id,n_slugs,n_prefixes,dominant_prefix,dominant_provenance,n_countries,country_coverage_pct,geographic_type,primary_region,primary_region_pct,label
,Int64,Int64,Int64,String7,String,Int64,Float64,String?,String?,Float64?,String
1,1,33,3,sgi,SURVEY,51,25.5,multi-regional,Europe,62.7,Mixed: sgi+ (33 slugs)
2,2,28,4,br,SURVEY,193,96.5,global,Africa,28.0,Global SURVEY (br+)
3,3,58,3,aii,SURVEY,58,29.0,regional,Africa,91.4,Africa-focused (aii+)
4,4,26,1,aii,EXPERT,11,5.5,regional,Africa,100.0,aii
5,5,77,19,iaep,SURVEY,195,97.5,global,Africa,27.7,Global SURVEY (iaep+)
6,6,22,4,bl,SURVEY,156,78.0,sparse,Africa,27.6,Mixed: bl+ (22 slugs)
7,7,65,6,bti,SURVEY,165,82.5,global,Africa,31.5,Global SURVEY (bti+)
8,8,19,4,ident,QoG Standard,200,100.0,global,Africa,27.0,Global QoG Standard (ident+)
9,9,125,11,wdi,SURVEY,194,97.0,global,Africa,27.8,Global SURVEY (wdi+)


## ht_region Alignment

In [10]:
result.ht_validation

Row,cluster_id,best_ht_region,ht_region_jaccard,n_countries
,Int64,Int64,Float64,Int64
1,1,5,0.444,51
2,2,4,0.254,193
3,3,4,0.814,58
4,4,4,0.224,11
5,5,4,0.251,195
6,6,4,0.228,156
7,7,4,0.281,165
8,8,4,0.245,200
9,9,4,0.253,194


## Explore a Specific Cluster

Change `cid` to inspect different clusters.

In [7]:
cid = first(result.profiles.cluster_id)
cluster_slugs = filter(r -> r.cluster_id == cid, result.slug_clusters)
println("Cluster $cid: $(nrow(cluster_slugs)) slugs")
leftjoin(cluster_slugs, meta_df[:, [:slug, :prefix, :label, :provenance]], on=:slug)

LoadError: invalid assignment to constant Main.cluster_slugs. This redefinition may be permitted using the `const` keyword.

## Experiment with Thresholds

In [ ]:
# Tighter clusters: higher min_sim, lower k
# result_tight = run_slug_clustering(df, meta_df, geo_df;
#     presence_min_pct=0.15, min_sim=0.20, k=10)

# Looser clusters: lower min_sim, higher k
# result_loose = run_slug_clustering(df, meta_df, geo_df;
#     presence_min_pct=0.05, min_sim=0.05, k=30)

## Save Results

In [ ]:
# CSV.write("data/slug_clusters.csv", result.slug_clusters)
# println("✅ Saved slug_clusters.csv")